# Module 3: Utility Function Ranking

This notebook executes the full Module 3 ranking workflow and documents the result in report format.

Goal:
- take the Module 2 candidate set,
- compute the four calibrated utility components,
- combine them with the baseline weights,
- produce the final top-5 recommendations per user.

Reusable scoring logic lives in `src/utility_scorer.py`. The notebook focuses on orchestration, printed diagnostics, validation, and visual explanation.

Refresh marker: notebook updated from Codex during this session.

## Ranking Design

The unified utility function is:

$$
U(i,u) = w_1 \cdot Relevance(i,u) + w_2 \cdot Uplift(i,u) + w_3 \cdot Margin(i) + w_4 \cdot Context(i,u)
$$

Baseline weights:
- $w_1 = 0.40$ for relevance
- $w_2 = 0.25$ for uplift
- $w_3 = 0.20$ for margin
- $w_4 = 0.15$ for context

Interpretation:
- `Relevance` preserves behavioral fit.
- `Uplift` penalizes habitual sure-thing items.
- `Margin` aligns recommendations with retailer economics.
- `Context` adapts the ranker to current promotion conditions.

In [2]:
from pathlib import Path
import importlib.util
import sys

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data_loader import load_or_build_master_transactions
from src.utility_scorer import (
    DEFAULT_UTILITY_WEIGHTS,
    build_commodity_margin_table,
    score_candidate_set,
)

DATA_RAW = ROOT / 'data' / '01_raw'
DATA_PROCESSED = ROOT / 'data' / '02_processed'

def read_csv_or_none(path: Path):
    if not path.exists():
        return None
    frame = pd.read_csv(path)
    if list(frame.columns) == ['version https://git-lfs.github.com/spec/v1']:
        return None
    return frame

def package_available(name: str) -> bool:
    return importlib.util.find_spec(name) is not None

print('Project root:', ROOT)
print('implicit available:', package_available('implicit'))
print('mlxtend available:', package_available('mlxtend'))
print('plotly available:', package_available('plotly'))
print('Default weights:', DEFAULT_UTILITY_WEIGHTS)

Project root: D:\Project\Data_Driven_Marketing_complete-journey-main
implicit available: True
mlxtend available: True
plotly available: True
Default weights: {'relevance': 0.4, 'uplift': 0.25, 'margin': 0.2, 'context': 0.15}


## Inputs

Module 3 uses:
- `candidate_set_module2.csv` from the recall layer,
- `campaign_table.csv` and `campaign_desc.csv` for campaign state,
- `causal_data.csv` for promoted-item status,
- `commodity_margin.csv` for business alignment,
- transaction history to compute habit strength and deal sensitivity.

If a processed artifact is missing, the notebook rebuilds it from raw data when the required dependencies are available.

In [4]:
campaign_table = pd.read_csv(DATA_RAW / 'campaign_table.csv')
campaign_desc = pd.read_csv(DATA_RAW / 'campaign_desc.csv')
causal_data = pd.read_csv(DATA_RAW / 'causal_data.csv')
product_catalog = pd.read_csv(DATA_RAW / 'product.csv', usecols=['PRODUCT_ID', 'COMMODITY_DESC', 'BRAND'])
tx_all, tx_organic = load_or_build_master_transactions(DATA_RAW, DATA_PROCESSED)

candidate_set = read_csv_or_none(DATA_PROCESSED / 'candidate_set_module2.csv')
commodity_margin = read_csv_or_none(DATA_PROCESSED / 'commodity_margin.csv')

print('campaign_table rows:', len(campaign_table))
print('campaign_desc rows:', len(campaign_desc))
print('causal_data rows:', len(causal_data))
print('product rows:', len(product_catalog))
print('tx_all rows:', len(tx_all))
print('tx_organic rows:', len(tx_organic))
print('candidate_set ready:', candidate_set is not None)
print('commodity_margin ready:', commodity_margin is not None)
display(campaign_desc.head())
display(tx_all.head())

campaign_table rows: 7208
campaign_desc rows: 30
causal_data rows: 36786524
product rows: 92353
tx_all rows: 2595732
tx_organic rows: 96240
candidate_set ready: True
commodity_margin ready: True


DESCRIPTION  CAMPAIGN  START_DAY  END_DAY
      TypeB        24        659      719
      TypeC        15        547      708
      TypeB        25        659      691
      TypeC        20        615      685
      TypeB        23        646      684


 household_key   BASKET_ID  DAY  PRODUCT_ID  QUANTITY  SALES_VALUE  RETAIL_DISC  WEEK_NO  COUPON_DISC  COUPON_MATCH_DISC              COMMODITY_DESC  Is_Promoted_Item  Revenue_Retailer
          2375 26984851472    1     1004906         1         1.39         0.60        1          0.0                0.0                    POTATOES              True              1.39
          2375 26984851472    1     1033142         1         0.82         0.00        1          0.0                0.0                      ONIONS             False              0.82
          2375 26984851472    1     1036325         1         0.99         0.30        1          0.0                0.0     VEGETABLES - ALL OTHERS              True              0.99
          2375 26984851472    1     1082185         1         1.21         0.00        1          0.0                0.0              TROPICAL FRUIT             False              1.21
          2375 26984851472    1     8160430         1         1.50         

In [5]:
if commodity_margin is None:
    print('commodity_margin.csv is not materialized locally. Rebuilding from product brand proxies.')
    commodity_margin = build_commodity_margin_table(product_catalog)
    commodity_margin.to_csv(DATA_PROCESSED / 'commodity_margin.csv', index=False)
    print('saved:', DATA_PROCESSED / 'commodity_margin.csv')
else:
    print('Using existing commodity_margin.csv')

display(commodity_margin.head(10))
print('commodity_margin commodities:', commodity_margin['COMMODITY_DESC'].nunique())

Using existing commodity_margin.csv
commodity_margin commodities: 308


      COMMODITY_DESC  Raw_Margin  Normalized_Margin
                             0.2                0.0
     (CORP USE ONLY)         0.4                1.0
  ADULT INCONTINENCE         0.4                1.0
            AIR CARE         0.4                1.0
          ANALGESICS         0.4                1.0
            ANTACIDS         0.4                1.0
             APPAREL         0.4                1.0
              APPLES         0.4                1.0
AUDIO/VIDEO PRODUCTS         0.4                1.0
 AUTOMOTIVE PRODUCTS         0.4                1.0


In [6]:
if candidate_set is None:
    print('candidate_set_module2.csv is not materialized locally.')
    if not package_available('implicit') or not package_available('mlxtend'):
        raise RuntimeError(
            'Cannot rebuild candidate_set locally because implicit/mlxtend are not installed in this environment.'
        )
    from src.recall_engine import build_als_model, build_candidate_set, build_mba_rules
    print('Rebuilding Module 2 candidate set from raw data...')
    mba_rules_long, _ = build_mba_rules(tx_organic)
    model_artifacts = build_als_model(tx_all)
    _, _, user_to_idx, _, _, _, users, items, user_factors, item_factors = model_artifacts
    candidate_artifacts = build_candidate_set(
        tx_all=tx_all,
        mba_rules_long=mba_rules_long,
        users=users,
        user_to_idx=user_to_idx,
        user_factors=user_factors,
        item_factors=item_factors,
        items=items,
    )
    candidate_set = candidate_artifacts.candidate_set
    candidate_set.to_csv(DATA_PROCESSED / 'candidate_set_module2.csv', index=False)
    print('saved:', DATA_PROCESSED / 'candidate_set_module2.csv')
else:
    print('Using existing candidate_set_module2.csv')

print('candidate rows:', len(candidate_set))
print('candidate users:', candidate_set['household_key'].nunique())
display(candidate_set.head(10))

Using existing candidate_set_module2.csv
candidate rows: 115583
candidate users: 2500


 household_key         COMMODITY_DESC                                      seed_items  relevance_als  relevance_mba  Relevance source source_detail  snapshot_week  was_recently_purchased
             1      BAKED SWEET GOODS CITRUS | SHORTENING/OIL | ORAL HYGIENE PRODUCTS       0.075771       0.000000   0.075771    als           ALS            102                   False
             1           BAKING MIXES CITRUS | SHORTENING/OIL | ORAL HYGIENE PRODUCTS       0.311801       0.000000   0.311801    als           ALS            102                   False
             1           BAKING NEEDS CITRUS | SHORTENING/OIL | ORAL HYGIENE PRODUCTS       0.211639       0.000000   0.211639    als           ALS            102                   False
             1             BEERS/ALES CITRUS | SHORTENING/OIL | ORAL HYGIENE PRODUCTS       0.000000       0.600756   0.600756    mba           MBA            102                   False
             1                 BLEACH CITRUS | SHORTENING/OIL | O

## Step 3.1: Relevance

Definition:

$$
Relevance(i,u) = \max(Relevance_{ALS}(i,u), Relevance_{MBA}(i,u))
$$

Rationale:
- both Module 2 signals are already normalized to $[0,1]$,
- the stronger signal is trusted for each user-item pair.

In [8]:
step31 = candidate_set[['household_key', 'COMMODITY_DESC', 'relevance_als', 'relevance_mba']].copy()
step31['Relevance'] = step31[['relevance_als', 'relevance_mba']].max(axis=1)

print('Step 3.1 completed')
print('relevance min/max:', float(step31['Relevance'].min()), float(step31['Relevance'].max()))
display(step31.head(15))

Step 3.1 completed
relevance min/max: 0.0 1.0


 household_key         COMMODITY_DESC  relevance_als  relevance_mba  Relevance
             1      BAKED SWEET GOODS       0.075771       0.000000   0.075771
             1           BAKING MIXES       0.311801       0.000000   0.311801
             1           BAKING NEEDS       0.211639       0.000000   0.211639
             1             BEERS/ALES       0.000000       0.600756   0.600756
             1                 BLEACH       0.629424       0.000000   0.629424
             1       BREAKFAST SWEETS       0.136995       0.000000   0.136995
             1    CHRISTMAS  SEASONAL       0.460816       0.000000   0.460816
             1    DISHWASH DETERGENTS       0.154000       0.000000   0.154000
             1 DRY BN/VEG/POTATO/RICE       0.119236       0.000000   0.119236
             1      DRY NOODLES/PASTA       0.070850       0.000000   0.070850
             1           FROZEN PIZZA       0.022360       0.000000   0.022360
             1 FRZN MEAT/MEAT DINNERS       0.197586

## Step 3.2: Uplift

Definition:

$$
Habit\_Strength(i,u) = \frac{\#\;baskets\;with\;item\;i}{\#\;total\;baskets\;for\;user\;u}
$$

$$
Uplift(i,u) = 1 - Habit\_Strength(i,u)
$$

Interpretation:
- habitual commodities receive a lower uplift score,
- unfamiliar but plausible commodities receive a higher uplift score.

In [10]:
user_baskets = tx_all.groupby('household_key')['BASKET_ID'].nunique().rename('Total_Baskets').reset_index()
item_baskets = tx_all.groupby(['household_key', 'COMMODITY_DESC'])['BASKET_ID'].nunique().rename('Item_Baskets').reset_index()
step32 = step31.merge(item_baskets, on=['household_key', 'COMMODITY_DESC'], how='left')
step32 = step32.merge(user_baskets, on='household_key', how='left')
step32['Item_Baskets'] = step32['Item_Baskets'].fillna(0)
step32['Habit_Strength'] = (step32['Item_Baskets'] / step32['Total_Baskets'].replace(0, pd.NA)).fillna(0).clip(0, 1)
step32['Uplift'] = 1.0 - step32['Habit_Strength']

print('Step 3.2 completed')
print('habit strength min/max:', float(step32['Habit_Strength'].min()), float(step32['Habit_Strength'].max()))
print('uplift min/max:', float(step32['Uplift'].min()), float(step32['Uplift'].max()))
display(step32[['household_key', 'COMMODITY_DESC', 'Item_Baskets', 'Total_Baskets', 'Habit_Strength', 'Uplift']].head(15))

Step 3.2 completed
habit strength min/max: 0.0 1.0
uplift min/max: 0.0 1.0


 household_key         COMMODITY_DESC  Item_Baskets  Total_Baskets  Habit_Strength   Uplift
             1      BAKED SWEET GOODS           4.0             86        0.046512 0.953488
             1           BAKING MIXES           7.0             86        0.081395 0.918605
             1           BAKING NEEDS           6.0             86        0.069767 0.930233
             1             BEERS/ALES           0.0             86        0.000000 1.000000
             1                 BLEACH           5.0             86        0.058140 0.941860
             1       BREAKFAST SWEETS          10.0             86        0.116279 0.883721
             1    CHRISTMAS  SEASONAL           2.0             86        0.023256 0.976744
             1    DISHWASH DETERGENTS           9.0             86        0.104651 0.895349
             1 DRY BN/VEG/POTATO/RICE           2.0             86        0.023256 0.976744
             1      DRY NOODLES/PASTA          14.0             86        0.1627

## Step 3.3: Margin

Definition:
- use the commodity-level `Normalized_Margin` score from the business proxy table,
- values are min-max scaled to $[0,1]$.

Interpretation:
- high-margin categories contribute more to the final rank,
- missing commodities default to `0.0` so the pipeline stays stable.

In [12]:
step33 = step32.merge(commodity_margin[['COMMODITY_DESC', 'Normalized_Margin']], on='COMMODITY_DESC', how='left')
step33['Normalized_Margin'] = step33['Normalized_Margin'].fillna(0.0).clip(0, 1)

print('Step 3.3 completed')
print('margin-covered share:', round((step33['Normalized_Margin'] > 0).mean(), 4))
print('margin min/max:', float(step33['Normalized_Margin'].min()), float(step33['Normalized_Margin'].max()))
display(step33[['household_key', 'COMMODITY_DESC', 'Normalized_Margin']].head(15))

Step 3.3 completed
margin-covered share: 0.873
margin min/max: 0.0 1.0


 household_key         COMMODITY_DESC  Normalized_Margin
             1      BAKED SWEET GOODS                1.0
             1           BAKING MIXES                1.0
             1           BAKING NEEDS                1.0
             1             BEERS/ALES                1.0
             1                 BLEACH                1.0
             1       BREAKFAST SWEETS                1.0
             1    CHRISTMAS  SEASONAL                1.0
             1    DISHWASH DETERGENTS                1.0
             1 DRY BN/VEG/POTATO/RICE                1.0
             1      DRY NOODLES/PASTA                1.0
             1           FROZEN PIZZA                1.0
             1 FRZN MEAT/MEAT DINNERS                1.0
             1          FRZN POTATOES                1.0
             1                 GRAPES                0.0
             1 HOUSEHOLD CLEANG NEEDS                1.0


## Step 3.4: Context

Context combines:
- household deal sensitivity,
- active campaign status,
- whether the commodity is currently promoted.

Logic summary:
- deal-sensitive user + active campaign + promoted item -> `1.0`
- deal-sensitive user + active campaign + full-price item -> `0.5`
- low deal sensitivity + promoted item -> `0.2`
- promoted item without campaign urgency -> `0.5`
- otherwise -> `0.7`

In [14]:
basket_promo = tx_all.groupby(['household_key', 'BASKET_ID'])['Is_Promoted_Item'].any().reset_index(name='Basket_Has_Promo')
deal_sensitivity = basket_promo.groupby('household_key').agg(
    Promo_Baskets=('Basket_Has_Promo', 'sum'),
    Total_Baskets=('Basket_Has_Promo', 'size'),
).reset_index()
deal_sensitivity['Deal_Sensitivity'] = (
    deal_sensitivity['Promo_Baskets'] / deal_sensitivity['Total_Baskets'].replace(0, pd.NA)
).fillna(0).clip(0, 1)

snapshot_day = int(tx_all['DAY'].max())
snapshot_week = int(tx_all['WEEK_NO'].max()) if 'WEEK_NO' in tx_all.columns else None
active_campaign_ids = campaign_desc.loc[(campaign_desc['START_DAY'] <= snapshot_day) & (campaign_desc['END_DAY'] >= snapshot_day), 'CAMPAIGN'].drop_duplicates()
active_households = campaign_table[campaign_table['CAMPAIGN'].isin(active_campaign_ids)][['household_key']].drop_duplicates().assign(Is_Active_Campaign_Period=True)

current_promos = causal_data[pd.to_numeric(causal_data['WEEK_NO'], errors='coerce') == snapshot_week].copy()
current_promos['display_num'] = pd.to_numeric(current_promos['display'], errors='coerce').fillna(0)
current_promos['mailer_flag'] = ~current_promos['mailer'].fillna('0').astype(str).str.upper().isin(['', '0', 'FALSE', 'NAN', 'NONE', 'NO'])
current_promos['Item_Is_Promoted'] = (current_promos['display_num'] > 0) | current_promos['mailer_flag']
promoted_commodities = current_promos[current_promos['Item_Is_Promoted']][['PRODUCT_ID']].drop_duplicates().merge(
    product_catalog[['PRODUCT_ID', 'COMMODITY_DESC']], on='PRODUCT_ID', how='left'
).dropna(subset=['COMMODITY_DESC'])[['COMMODITY_DESC']].drop_duplicates().assign(Item_Is_Promoted=True)

step34 = step33.merge(deal_sensitivity[['household_key', 'Deal_Sensitivity']], on='household_key', how='left')
step34 = step34.merge(active_households, on='household_key', how='left')
step34 = step34.merge(promoted_commodities, on='COMMODITY_DESC', how='left')
step34['Deal_Sensitivity'] = step34['Deal_Sensitivity'].fillna(0.0).clip(0, 1)
step34['Is_Active_Campaign_Period'] = step34['Is_Active_Campaign_Period'].astype('boolean').fillna(False).astype(bool)
step34['Item_Is_Promoted'] = step34['Item_Is_Promoted'].astype('boolean').fillna(False).astype(bool)

def context_rule(deal_sensitivity_value, is_active_campaign, item_is_promoted):
    if deal_sensitivity_value > 0.6 and is_active_campaign and item_is_promoted:
        return 1.0
    if deal_sensitivity_value > 0.6 and is_active_campaign and not item_is_promoted:
        return 0.5
    if deal_sensitivity_value < 0.3 and item_is_promoted:
        return 0.2
    if not is_active_campaign and item_is_promoted:
        return 0.5
    return 0.7

step34['Context'] = [
    context_rule(ds, active, promoted)
    for ds, active, promoted in step34[['Deal_Sensitivity', 'Is_Active_Campaign_Period', 'Item_Is_Promoted']].itertuples(index=False, name=None)
]

print('Step 3.4 completed')
print('snapshot_day:', snapshot_day)
print('snapshot_week:', snapshot_week)
print('active campaigns:', len(active_campaign_ids))
print('active households:', len(active_households))
print('promoted commodities this week:', len(promoted_commodities))
display(deal_sensitivity.head(10))
display(step34[['household_key', 'COMMODITY_DESC', 'Deal_Sensitivity', 'Is_Active_Campaign_Period', 'Item_Is_Promoted', 'Context']].head(15))

Step 3.4 completed
snapshot_day: 711
snapshot_week: 102
active campaigns: 1
active households: 100
promoted commodities this week: 0


 household_key  Promo_Baskets  Total_Baskets  Deal_Sensitivity
             1             70             86          0.813953
             2             42             45          0.933333
             3             46             47          0.978723
             4             28             30          0.933333
             5             30             40          0.750000
             6            203            250          0.812000
             7             55             59          0.932203
             8            104            113          0.920354
             9             18             20          0.900000
            10              6              9          0.666667


 household_key         COMMODITY_DESC  Deal_Sensitivity  Is_Active_Campaign_Period  Item_Is_Promoted  Context
             1      BAKED SWEET GOODS          0.813953                      False             False      0.7
             1           BAKING MIXES          0.813953                      False             False      0.7
             1           BAKING NEEDS          0.813953                      False             False      0.7
             1             BEERS/ALES          0.813953                      False             False      0.7
             1                 BLEACH          0.813953                      False             False      0.7
             1       BREAKFAST SWEETS          0.813953                      False             False      0.7
             1    CHRISTMAS  SEASONAL          0.813953                      False             False      0.7
             1    DISHWASH DETERGENTS          0.813953                      False             False      0.7
          

## Step 3.5: Unified Utility

Final score:

$$
U(i,u)=0.40 \cdot Relevance + 0.25 \cdot Uplift + 0.20 \cdot Margin + 0.15 \cdot Context
$$

The output of this stage is the ranked candidate set plus the top 5 items per user.

In [16]:
w1 = DEFAULT_UTILITY_WEIGHTS['relevance']
w2 = DEFAULT_UTILITY_WEIGHTS['uplift']
w3 = DEFAULT_UTILITY_WEIGHTS['margin']
w4 = DEFAULT_UTILITY_WEIGHTS['context']

step35 = step34.copy()
step35['Utility'] = (
    w1 * step35['Relevance']
    + w2 * step35['Uplift']
    + w3 * step35['Normalized_Margin']
    + w4 * step35['Context']
)

top5_manual = step35.sort_values(
    ['household_key', 'Utility', 'Relevance', 'Normalized_Margin', 'COMMODITY_DESC'],
    ascending=[True, False, False, False, True],
).groupby('household_key', as_index=False).head(5).reset_index(drop=True)

print('Step 3.5 completed')
print('utility min/max:', float(step35['Utility'].min()), float(step35['Utility'].max()))
print('top5 rows:', len(top5_manual))
display(top5_manual[['household_key', 'COMMODITY_DESC', 'Relevance', 'Uplift', 'Normalized_Margin', 'Context', 'Utility']].head(20))

Step 3.5 completed
utility min/max: 0.16909093049180643 0.9550000000000001
top5 rows: 12500


 household_key          COMMODITY_DESC  Relevance   Uplift  Normalized_Margin  Context  Utility
             1 VEGETABLES - ALL OTHERS   1.000000 0.895349                1.0      0.7 0.928837
             1                 POPCORN   0.845775 0.988372                1.0      0.7 0.890403
             1              BEERS/ALES   0.600756 1.000000                1.0      0.7 0.795303
             1                  BLEACH   0.629424 0.941860                1.0      0.7 0.792235
             1                  GRAPES   1.000000 0.953488                0.0      0.7 0.743372
             2 VEGETABLES - ALL OTHERS   1.000000 0.977778                1.0      0.7 0.949444
             2      HAIR CARE PRODUCTS   1.000000 0.866667                1.0      0.7 0.921667
             2                  APPLES   1.000000 0.844444                1.0      0.7 0.916111
             2                  ONIONS   0.954906 0.911111                1.0      0.7 0.914740
             2    MAKEUP AND TREATMENT  

## Validate Against `src`

The notebook-level derivation above is kept for transparency. This section reruns the same logic through the reusable `src` implementation and uses that as the persisted output.

In [18]:
utility_artifacts = score_candidate_set(
    candidate_set=candidate_set,
    history=tx_all,
    commodity_margin=commodity_margin,
    campaign_table=campaign_table,
    campaign_desc=campaign_desc,
    causal_data=causal_data,
    product_lookup=product_catalog[['PRODUCT_ID', 'COMMODITY_DESC']],
    weights=DEFAULT_UTILITY_WEIGHTS,
    top_k=5,
)

scored_candidates = utility_artifacts.scored_candidates.copy()
top5 = utility_artifacts.top_recommendations.copy()

print('src scored rows:', len(scored_candidates))
print('src top5 rows:', len(top5))
display(scored_candidates.head(10))
display(top5.head(20))

src scored rows: 115583
src top5 rows: 12500


 household_key          COMMODITY_DESC                                      seed_items  relevance_als  relevance_mba  Relevance source source_detail  snapshot_week  was_recently_purchased  habit_strength   Uplift  Normalized_Margin  deal_sensitivity  is_active_campaign_period  item_is_promoted  Context  Utility
             1 VEGETABLES - ALL OTHERS CITRUS | SHORTENING/OIL | ORAL HYGIENE PRODUCTS       0.000000       1.000000   1.000000    mba           MBA            102                   False        0.104651 0.895349                1.0          0.813953                      False             False      0.7 0.928837
             1                 POPCORN CITRUS | SHORTENING/OIL | ORAL HYGIENE PRODUCTS       0.845775       0.000000   0.845775    als           ALS            102                   False        0.011628 0.988372                1.0          0.813953                      False             False      0.7 0.890403
             1              BEERS/ALES CITRUS | SHORTENING/OI

 household_key          COMMODITY_DESC                                                                             seed_items  relevance_als  relevance_mba  Relevance source source_detail  snapshot_week  was_recently_purchased  habit_strength   Uplift  Normalized_Margin  deal_sensitivity  is_active_campaign_period  item_is_promoted  Context  Utility
             1 VEGETABLES - ALL OTHERS                                        CITRUS | SHORTENING/OIL | ORAL HYGIENE PRODUCTS       0.000000       1.000000   1.000000    mba           MBA            102                   False        0.104651 0.895349                1.0          0.813953                      False             False      0.7 0.928837
             1                 POPCORN                                        CITRUS | SHORTENING/OIL | ORAL HYGIENE PRODUCTS       0.845775       0.000000   0.845775    als           ALS            102                   False        0.011628 0.988372                1.0          0.813953        

## Plotly Visuals

Three visuals are added because they explain the ranking clearly:
- a household-level stacked contribution chart answers *why* the top items won,
- a component comparison summarizes how the ranker shifts candidates into the final top 5,
- a relevance-uplift landscape shows the tradeoff surface the utility function is navigating.

This is a better fit than generic histograms alone because the project is fundamentally about explainable ranking decisions.

In [20]:
scored_candidates['Relevance_Contribution'] = w1 * scored_candidates['Relevance']
scored_candidates['Uplift_Contribution'] = w2 * scored_candidates['Uplift']
scored_candidates['Margin_Contribution'] = w3 * scored_candidates['Normalized_Margin']
scored_candidates['Context_Contribution'] = w4 * scored_candidates['Context']

top5['Relevance_Contribution'] = w1 * top5['Relevance']
top5['Uplift_Contribution'] = w2 * top5['Uplift']
top5['Margin_Contribution'] = w3 * top5['Normalized_Margin']
top5['Context_Contribution'] = w4 * top5['Context']

example_household = int(top5.groupby('household_key')['Utility'].max().idxmax()) if 'household_key' in top5.columns else int(top5['household_key'].iloc[0])
example_household = int(top5['household_key'].iloc[0])
example_top5 = top5[top5['household_key'] == example_household].copy()
print('example household for decomposition:', example_household)
display(example_top5[['household_key', 'COMMODITY_DESC', 'Utility']])

example household for decomposition: 1


 household_key          COMMODITY_DESC  Utility
             1 VEGETABLES - ALL OTHERS 0.928837
             1                 POPCORN 0.890403
             1              BEERS/ALES 0.795303
             1                  BLEACH 0.792235
             1                  GRAPES 0.743372


In [21]:
decomp_long = example_top5.melt(
    id_vars=['COMMODITY_DESC', 'Utility'],
    value_vars=[
        'Relevance_Contribution',
        'Uplift_Contribution',
        'Margin_Contribution',
        'Context_Contribution',
    ],
    var_name='Component',
    value_name='Contribution',
)

fig_decomp = px.bar(
    decomp_long,
    x='Contribution',
    y='COMMODITY_DESC',
    color='Component',
    orientation='h',
    barmode='stack',
    title=f'Utility Decomposition for Household {example_household}',
    category_orders={'COMMODITY_DESC': example_top5.sort_values('Utility')['COMMODITY_DESC'].tolist()},
)
fig_decomp.update_layout(height=450, xaxis_title='Weighted Utility Contribution', yaxis_title='Commodity')
fig_decomp.show()

Plotly Figure: Utility Decomposition for Household 1

In [22]:
component_compare = pd.DataFrame(
    {
        'Group': ['All Scored Candidates', 'Top 5 Recommendations'],
        'Relevance': [scored_candidates['Relevance'].mean(), top5['Relevance'].mean()],
        'Uplift': [scored_candidates['Uplift'].mean(), top5['Uplift'].mean()],
        'Margin': [scored_candidates['Normalized_Margin'].mean(), top5['Normalized_Margin'].mean()],
        'Context': [scored_candidates['Context'].mean(), top5['Context'].mean()],
        'Utility': [scored_candidates['Utility'].mean(), top5['Utility'].mean()],
    }
)
component_compare_long = component_compare.melt(id_vars='Group', var_name='Metric', value_name='Average Score')

fig_compare = px.bar(
    component_compare_long,
    x='Metric',
    y='Average Score',
    color='Group',
    barmode='group',
    title='Average Component Scores: Full Candidate Set vs Top 5',
)
fig_compare.update_layout(height=420)
fig_compare.show()

Plotly Figure: Average Component Scores: Full Candidate Set vs Top 5

In [23]:
sample_size = min(4000, len(scored_candidates))
scatter_sample = scored_candidates.sample(sample_size, random_state=42).copy()
scatter_sample['Source'] = scatter_sample['source_detail'].fillna('UNKNOWN')

fig_scatter = px.scatter(
    scatter_sample,
    x='Relevance',
    y='Uplift',
    color='Utility',
    symbol='Source',
    size='Normalized_Margin',
    hover_data=['household_key', 'COMMODITY_DESC', 'Context'],
    title='Recommendation Landscape: Relevance vs Uplift',
)
fig_scatter.update_layout(height=520)
fig_scatter.show()

top5_source_mix = top5['source_detail'].value_counts().rename_axis('Source').reset_index(name='Count')
fig_source = px.bar(
    top5_source_mix,
    x='Source',
    y='Count',
    color='Source',
    title='Source Mix in Final Top 5 Recommendations',
)
fig_source.update_layout(height=360, showlegend=False)
fig_source.show()

Plotly Figure: Recommendation Landscape: Relevance vs Uplift

Plotly Figure: Source Mix in Final Top 5 Recommendations

## Persist Outputs

The scored candidate set and top-5 recommendation table are saved for downstream validation, reporting, and dashboard work.

In [25]:
scored_path = DATA_PROCESSED / 'candidate_set_module3_scored.csv'
top5_path = DATA_PROCESSED / 'top5_recommendations_module3.csv'

scored_candidates.to_csv(scored_path, index=False)
top5.to_csv(top5_path, index=False)

print('saved scored output:', scored_path)
print('saved top5 output:', top5_path)

saved scored output: D:\Project\Data_Driven_Marketing_complete-journey-main\data\02_processed\candidate_set_module3_scored.csv
saved top5 output: D:\Project\Data_Driven_Marketing_complete-journey-main\data\02_processed\top5_recommendations_module3.csv
